# Validação — antes/depois na amostra (S18)

Protocolo e plano de análise fixados no **ADR 004** (`docs/decisoes/004-amostra-e-protocolo-da-validacao.md`) antes das medições.

- **Entrada:** `data/processed/validation.csv` (gerado por `npm run validate -- run --sample data/validation/amostra.csv`; uma linha por site, com original, corrigido e delta de cada desfecho) e `data/validation/amostra.csv`.
- **Comparação:** cópia local original × cópia local corrigida (regra 7, ADR 018), mesmo servidor, Lighthouse 3× por variante (mediana pela nota, ADR 009). Delta = corrigido − original.
- **Teste:** Wilcoxon pareado (*signed-rank*), bilateral, α = 0,05; Holm na família principal (nota de performance, LCP, CLS, TBT, índice de semântica). Pares com delta 0 saem do teste (`zero_method="wilcox"`) e o n efetivo é informado. Efeito: mediana das diferenças e correlação rank-bisserial pareada, r = (W⁺ − W⁻) / (W⁺ + W⁻).
- **Saídas:** `docs/resultados/tabelas/validacao-*.csv`, `docs/resultados/figuras/validacao-antes-depois.png` e `docs/resultados/tabelas/validacao-meta.json` (regra 10).
- n = 12: resultados exploratórios (ADR 004, limitações).

Executar da raiz do repo: `python -m jupyter nbconvert --to notebook --execute --inplace analysis/validacao.ipynb`.

In [1]:
import hashlib, json, platform, subprocess
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
VALIDACAO = ROOT / "data" / "processed" / "validation.csv"
AMOSTRA = ROOT / "data" / "validation" / "amostra.csv"
TAB = ROOT / "docs" / "resultados" / "tabelas"
FIG = ROOT / "docs" / "resultados" / "figuras"
TAB.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

ALFA = 0.05
# ADR 004, 7: família principal (Holm). Sentido de "melhor" de cada desfecho.
PRINCIPAIS = {"nota_performance": "maior", "lcp_ms": "menor", "cls": "menor", "tbt_ms": "menor", "indice_semantica": "maior"}

def git(*args):
    try:
        return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return None

sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
meta = {
    "generatedAt": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "toolVersions": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__, "scipy": scipy.__version__, "matplotlib": matplotlib.__version__},
    "gitCommit": git("rev-parse", "HEAD"),
    "entrada": {
        "validation": {"arquivo": "data/processed/validation.csv", "sha256": sha(VALIDACAO)},
        "amostra": {"arquivo": "data/validation/amostra.csv", "sha256": sha(AMOSTRA)},
    },
}
meta

{'generatedAt': '2026-11-16T19:10:16+00:00',
 'toolVersions': {'python': '3.13.5',
  'pandas': '2.3.1',
  'numpy': '2.2.6',
  'scipy': '1.16.3',
  'matplotlib': '3.10.5'},
 'gitCommit': '107f56e3ef3e38e7406db9e7ef9ab0d48a1ce060',
 'entrada': {'validation': {'arquivo': 'data/processed/validation.csv',
   'sha256': 'e120477b686a19edbf9302ac8173614e39ca3f58d0a12e7fa370a04856d9c98f'},
  'amostra': {'arquivo': 'data/validation/amostra.csv',
   'sha256': '04f408ebd7957f15acb288ace768d917965403ef3c0a8acbfbcd15a2fa5fb663'}}}

In [2]:
v = pd.read_csv(VALIDACAO)
amostra = pd.read_csv(AMOSTRA)
faltando = sorted(set(amostra.id) - set(v.id))
print("sites na amostra:", len(amostra), "| validados:", len(v), "| faltando:", faltando)
cols = ["id", "grupo", "correcoes", "correcoes_llm", "faltas_original", "faltas_corrigido", "amplitude_original", "amplitude_corrigido",
        "nota_performance_original", "nota_performance_corrigido", "nota_performance_delta",
        "indice_semantica_original", "indice_semantica_corrigido", "indice_semantica_delta"]
por_site = v[cols].sort_values(["grupo", "id"]).reset_index(drop=True)
por_site.to_csv(TAB / "validacao-por-site.csv", index=False)
por_site

sites na amostra: 12 | validados: 12 | faltando: []


,id,grupo,correcoes,correcoes_llm,faltas_original,faltas_corrigido,amplitude_original,amplitude_corrigido,nota_performance_original,nota_performance_corrigido,nota_performance_delta,indice_semantica_original,indice_semantica_corrigido,indice_semantica_delta
0,ce.ufrn.br,estatico,20,0,15,15,19,14,41,35,-6,59.007817,59.007817,0.000000
1,espacos-denf.imd.ufrn.br,estatico,4,0,2,2,0,16,42,42,0,37.593985,42.857143,5.263158
2,materialpublic.imd.ufrn.br,estatico,8,2,2,2,2,1,67,68,1,37.429775,56.460674,19.030899
3,agir.ufrn.br,framework-js,136,0,39,39,4,1,57,58,1,49.644719,54.907877,5.263158
4,entbl2020.ccsa.ufrn.br,framework-js,1,0,8,8,0,0,45,45,0,57.943245,63.206403,5.263158
5,semanadeturismo.ccsa.ufrn.br,framework-js,1,0,8,8,3,1,48,48,0,46.597714,51.860872,5.263158
6,observatoriodasdesigualdades.ccsa.ufrn.br,outro,9,0,64,64,10,6,81,81,0,73.419878,73.419878,0.000000
7,periodicos.ufrn.br,outro,36,0,12,12,3,1,65,66,1,60.441086,60.441086,0.000000
8,ria.ufrn.br,outro,14,0,3,3,3,7,84,85,1,61.729906,66.993064,5.263158
9,3cnienssup.ufrn.br,wordpress,4,0,33,33,0,0,48,67,19,60.352891,60.352891,0.000000


## 1. Teste pareado na família principal

Para LCP, CLS e TBT, valores menores são melhores: um delta negativo é melhora.

In [3]:
def holm(p):
    p = np.asarray(p, dtype=float)
    ordem = np.argsort(p)
    m = len(p)
    aj = np.empty(m)
    acumulado = 0.0
    for k, i in enumerate(ordem):
        acumulado = max(acumulado, min(1.0, (m - k) * p[i]))
        aj[i] = acumulado
    return aj

def wilcoxon_pareado(delta):
    d = pd.Series(delta).dropna()
    nz = d[d != 0]
    linha = {"n_pares": len(d), "n_efetivo": len(nz), "melhorou": None, "piorou": None, "iguais": int((d == 0).sum()),
             "mediana_delta": float(d.median()) if len(d) else np.nan, "W": np.nan, "p": np.nan, "r_rank_bisserial": np.nan}
    if len(nz) == 0:
        return linha
    postos = stats.rankdata(nz.abs())
    w_mais, w_menos = postos[nz > 0].sum(), postos[nz < 0].sum()
    res = stats.wilcoxon(nz, zero_method="wilcox", alternative="two-sided")
    linha.update(W=float(res.statistic), p=float(res.pvalue), r_rank_bisserial=float((w_mais - w_menos) / (w_mais + w_menos)))
    return linha

linhas = []
for desf, melhor in PRINCIPAIS.items():
    d = v[f"{desf}_delta"]
    r = wilcoxon_pareado(d)
    sinal = 1 if melhor == "maior" else -1
    r["melhorou"] = int((sinal * d > 0).sum())
    r["piorou"] = int((sinal * d < 0).sum())
    linhas.append({"desfecho": desf, "melhor_se": melhor,
                   "mediana_original": float(v[f"{desf}_original"].median()), "mediana_corrigido": float(v[f"{desf}_corrigido"].median()), **r})
principal = pd.DataFrame(linhas)
testaveis = principal.p.notna()
principal["p_holm"] = np.nan
principal.loc[testaveis, "p_holm"] = holm(principal.loc[testaveis, "p"])
principal["significativo"] = principal.p_holm < ALFA
principal.to_csv(TAB / "validacao-wilcoxon.csv", index=False)
principal.round(4)

,desfecho,melhor_se,mediana_original,mediana_corrigido,n_pares,n_efetivo,melhorou,piorou,iguais,mediana_delta,W,p,r_rank_bisserial,p_holm,significativo
0,nota_performance,maior,54.0000,61.5000,12,7,5,2,5,0.0000,11.0,0.6875,0.2143,1.0000,False
1,lcp_ms,menor,9547.8050,9613.8394,12,12,8,4,0,-8.0492,28.0,0.4238,-0.2821,1.0000,False
2,cls,menor,0.0013,0.0005,12,5,4,1,7,0.0000,3.0,0.3125,-0.6000,1.0000,False
3,tbt_ms,menor,115.5000,118.7500,12,11,6,5,1,-0.1657,31.0,0.8799,0.0606,1.0000,False
4,indice_semantica,maior,59.1331,59.8056,12,7,6,1,5,2.6316,1.0,0.0312,0.9286,0.1562,False


## 2. Descritivas dos demais desfechos

Sem teste (ADR 004, 7): outras notas do Lighthouse, métricas de carga e as 8 subnotas do índice.

In [4]:
outros = [c[: -len("_delta")] for c in v.columns if c.endswith("_delta") and c[: -len("_delta")] not in PRINCIPAIS]
desc = pd.DataFrame([{
    "desfecho": k,
    "n": int(v[f"{k}_delta"].notna().sum()),
    "mediana_original": v[f"{k}_original"].median(),
    "mediana_corrigido": v[f"{k}_corrigido"].median(),
    "mediana_delta": v[f"{k}_delta"].median(),
    "delta_positivo": int((v[f"{k}_delta"] > 0).sum()),
    "delta_negativo": int((v[f"{k}_delta"] < 0).sum()),
    "delta_zero": int((v[f"{k}_delta"] == 0).sum()),
} for k in outros])
desc.to_csv(TAB / "validacao-descritivas.csv", index=False)
desc.round(3)

C:\Users\Mateo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\Mateo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\Mateo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,desfecho,n,mediana_original,mediana_corrigido,mediana_delta,delta_positivo,delta_negativo,delta_zero
0,nota_accessibility,12,77.500,83.000,2.500,6,0,6
1,nota_best_practices,12,92.000,88.000,0.000,0,4,8
2,nota_seo,12,82.500,82.500,0.000,0,0,12
3,fcp_ms,12,4168.058,4172.994,-0.708,5,7,0
4,speed_index_ms,12,4820.603,4734.671,-0.708,5,7,0
5,peso_total_bytes,12,1519102.500,1519136.000,9.500,7,5,0
6,requisicoes,12,49.000,49.500,0.000,1,5,6
7,indice_semantica_pesos_iguais,12,64.702,65.743,3.571,6,1,5
8,sub_landmarks,12,75.000,75.000,0.000,1,0,11
9,sub_titulos,12,50.000,50.000,0.000,0,0,12


## 3. Delta da nota de performance × variação entre execuções

A amplitude (máx − mín das 3 execuções de uma variante) é o ruído do Lighthouse no próprio site. Um delta menor que a amplitude não se distingue da variação entre execuções.

In [5]:
ruido = v[["id", "nota_performance_delta", "amplitude_original", "amplitude_corrigido"]].copy()
ruido["amplitude_max"] = ruido[["amplitude_original", "amplitude_corrigido"]].max(axis=1)
ruido["delta_acima_do_ruido"] = ruido.nota_performance_delta.abs() > ruido.amplitude_max
print("sites com |delta| acima da maior amplitude:", int(ruido.delta_acima_do_ruido.sum()), "de", len(ruido))
ruido.sort_values("nota_performance_delta")

sites com |delta| acima da maior amplitude: 1 de 12


,id,nota_performance_delta,amplitude_original,amplitude_corrigido,amplitude_max,delta_acima_do_ruido
1,ce.ufrn.br,-6,19,14,19,False
7,profletras.ufrn.br,-2,2,2,2,False
3,semanadeturismo.ccsa.ufrn.br,0,3,1,3,False
0,espacos-denf.imd.ufrn.br,0,0,16,16,False
5,entbl2020.ccsa.ufrn.br,0,0,0,0,False
8,mat.ufrn.br,0,1,3,3,False
10,observatoriodasdesigualdades.ccsa.ufrn.br,0,10,6,10,False
2,materialpublic.imd.ufrn.br,1,2,1,2,False
9,ria.ufrn.br,1,3,7,7,False
4,agir.ufrn.br,1,4,1,4,False


## 4. Figura antes/depois

Uma linha por site, da cópia original à corrigida. Azul: melhorou; laranja: piorou; cinza: igual.

In [6]:
COR = {"melhorou": "#2a78d6", "piorou": "#eb6834", "igual": "#8a8985"}
paineis = [("nota_performance", "Nota de performance (Lighthouse)"), ("indice_semantica", "Índice de semântica (ADR 003)")]
fig, eixos = plt.subplots(1, 2, figsize=(9, 4.6), sharey=False)
for ax, (desf, titulo) in zip(eixos, paineis):
    for _, s in v.iterrows():
        o, c = s[f"{desf}_original"], s[f"{desf}_corrigido"]
        if pd.isna(o) or pd.isna(c):
            continue
        estado = "igual" if c == o else ("melhorou" if c > o else "piorou")
        ax.plot([0, 1], [o, c], color=COR[estado], linewidth=2, marker="o", markersize=5, alpha=0.9)
    ax.set_xticks([0, 1], ["Original", "Corrigido"])
    ax.set_xlim(-0.25, 1.25)
    ax.set_ylim(0, 100)
    ax.set_title(titulo, fontsize=11)
    ax.grid(axis="y", color="#e4e3df", linewidth=0.8)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
from matplotlib.lines import Line2D
fig.legend(handles=[Line2D([0], [0], color=COR[k], linewidth=2, label=k) for k in COR], loc="lower center", ncol=3, frameon=False)
fig.suptitle(f"Cópia local original × corrigida (n = {len(v)} sites, ADR 004)", fontsize=12)
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig(FIG / "validacao-antes-depois.png", dpi=150)
plt.close(fig)
print("figura:", FIG / "validacao-antes-depois.png")

figura: C:\Users\Mateo\Documents\tcc-ufrn-web-audit\docs\resultados\figuras\validacao-antes-depois.png


In [7]:
meta["saidas"] = ["docs/resultados/tabelas/validacao-por-site.csv", "docs/resultados/tabelas/validacao-wilcoxon.csv",
                  "docs/resultados/tabelas/validacao-descritivas.csv", "docs/resultados/figuras/validacao-antes-depois.png"]
(TAB / "validacao-meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
meta["saidas"]

['docs/resultados/tabelas/validacao-por-site.csv',
 'docs/resultados/tabelas/validacao-wilcoxon.csv',
 'docs/resultados/tabelas/validacao-descritivas.csv',
 'docs/resultados/figuras/validacao-antes-depois.png']